# TMDB movie recommendation pipeline
This is the notebook that produces `movie_data.pkl` for both Streamlit interfaces. Run all cells from a fresh kernel. For a command-line rebuild, use `python build_model.py`. The generated pickle is intentionally excluded from Git.


In [ ]:
from pathlib import Path
import os
ROOT = Path.cwd()
if not (ROOT / "TMDB movie Dataset").is_dir():
    ROOT = ROOT.parent
os.chdir(ROOT)
import pandas as pd
import numpy as np
import ast

In [ ]:
credits = pd.read_csv('TMDB movie Dataset/tmdb_5000_credits.csv')
movies = pd.read_csv('TMDB movie Dataset/tmdb_5000_movies.csv')

In [ ]:
credits.head()

In [ ]:
movies.head(1)

In [ ]:
movies = movies.merge(credits, left_on='title', right_on='title')

In [ ]:
movies.head(1)

In [ ]:
movies = movies[['movie_id', 'title', 'overview', 'genres', 'keywords', 'cast', 'crew']]

In [ ]:
movies.head(1)

In [ ]:
def convert(obj):
    L = []
    for i in ast.literal_eval(obj):
        L.append(i['name'])
    return L

In [ ]:
movies['genres'] = movies['genres'].apply(convert)

In [ ]:
movies['genres']

In [ ]:
movies['keywords'] = movies['keywords'].apply(convert)

In [ ]:
movies['keywords']

In [ ]:
movies['cast'] = movies['cast'].apply(lambda x: [i['name'] for i in ast.literal_eval(x)[:3]])

In [ ]:
movies['crew'] = movies['crew'].apply(lambda x: [i['name'] for i in ast.literal_eval(x) if i['job'] == 'Director'])

In [ ]:
movies['tags'] = movies['genres'] + movies['keywords'] + movies['cast'] + movies['crew']

In [ ]:
movies['tags']

In [ ]:
movies['tags'] = movies['tags'].apply(lambda x: " ".join(x))

In [ ]:
movies['tags']

In [ ]:
movies = movies[['movie_id', 'title', 'overview', 'tags']]

In [ ]:
movies['tags'] = movies['tags'].apply(lambda x: x.lower())

In [ ]:
movies.head()

In [ ]:
from sklearn.feature_extraction.text import TfidfVectorizer
tfidf = TfidfVectorizer(stop_words='english')
tfidf_matrix = tfidf.fit_transform(movies['tags'])

In [ ]:
from sklearn.metrics.pairwise import cosine_similarity
cosine_sim = cosine_similarity(tfidf_matrix, tfidf_matrix)

In [ ]:
from movie_utils import recommend_movies

def get_recommendations(title, cosine_sim=cosine_sim):
    return recommend_movies(movies, cosine_sim, title)["title"]


In [ ]:
print(get_recommendations('The Dark Knight Rises'))

In [ ]:
import pickle
with open('movie_data.pkl', 'wb') as file:
    pickle.dump((movies, cosine_sim), file)